# Finding and downloading Landsat scenes (USGS M2M)

*Section 12 — Downloading data, notebook 1 of 8.* Previous: `11_case_study_species_distribution/02_species_distribution_modelling`. Next: `02_nasa_cmr_download_icesat2`.

**You will learn:**

- how to log in to the USGS Machine-to-Machine (M2M) API with `rsgispy.dataaccess.usgs_m2m`
- how to search for Landsat scenes by WRS-2 path/row, date and cloud cover
- how to request download URLs and keep track of downloads with a JSON database (`rsgispy.tools.http`)

**Before you start:** a USGS EROS Registration System (ERS) account (https://ers.cr.usgs.gov) with M2M access approved (request it at https://m2m.cr.usgs.gov). Store the username and password in an encoded credentials file (`usgs_userinfo.txt`, see *Username and password* below).

**Data:** downloaded from the USGS EarthExplorer M2M service into `outputs/`.

**Run time:** depends on the download; the search cells take about 1 minute.

> **Running this notebook:** every cell that logs in, queries the remote
> service or downloads data is tagged `skip-execution`, so the tutorial test
> runner does not run them. Run them yourself, in order, once you have an
> account and have created the credentials file.

This tutorial uses the USGS Machine-to-Machine (M2M) API to find Landsat
scenes and then download them.

To have access to the M2M service, you need to request access
(https://m2m.cr.usgs.gov/api/docs/json/).

In [1]:
import datetime
import os

import rsgispy.dataaccess.usgs_m2m
import rsgispy.tools.credentials
import rsgispy.tools.http

In [2]:
# Output folder for the downloads and the download database.
out_dir = "outputs"
os.makedirs(out_dir, exist_ok=True)

# The encoded username / password file (see below).
cred_file = "usgs_userinfo.txt"

## Username and password

Never write your username and password into a notebook. rsgispy provides a
simple encoding of the username and password so that they are not stored as
plain text. Note that the encoding (base64) is **not** secure: keep the file
private and never commit it to version control.

Create the encoded file with the `rsgisuserpassfile` command-line tool, which
is installed with rsgispy and prompts for the password:

`rsgisuserpassfile -o usgs_userinfo.txt -u <your USGS ERS username>`

or, from Python, with
`rsgispy.tools.credentials.create_username_password_file(username, password, out_file)`.
Once you have created `usgs_userinfo.txt` (in this folder, or change the path
below), read it into your notebook/script with `get_username_password`.
Alternatively, define the `RSGIS_USGS_USER` and `RSGIS_USGS_PASS` environment variables
and pass `None` for the username and password.

*RSGISLib users:* the tool was `rsgisuserpassfile.py <file>`; the output file
is now given with `-o`.

In [ ]:
username, password = rsgispy.tools.credentials.get_username_password(
    input_file=cred_file
)

## API key

For all M2M calls, you need to pass your API key, which is generated for your
session using your username and password. Note, the `RSGIS_USGS_USER` and
`RSGIS_USGS_PASS` environment variables can be defined for the system you are
using, and then you do not need to pass the username and password into the
function.

The API key is generated using the following function:

In [ ]:
api_key = rsgispy.dataaccess.usgs_m2m.usgs_login(username=username, password=password)

## Search by WRS-2 path/row

I would commonly search by Landsat WRS-2 path and row, and the way to do that
is to find the lat/long at the centre of the path/row of interest.
Alternatively, you can just search by a bounding box or point.

For this example, we'll search for row 24 / path 204, which is one of the
scenes which covers Aberystwyth.

In [ ]:
lon, lat = rsgispy.dataaccess.usgs_m2m.get_wrs_pt(
    api_key=api_key, row=24, path=204, grid_version=2
)
print(f"{lon}, {lat}")

## Search for scenes

The datasets which are available are listed below:

| Dataset Name | Dataset ID |
|-|-|
| Landsat 4/5 TM Collection 2 Level 1 | `landsat_tm_c2_l1` |
| Landsat 4/5 TM Collection 2 Level 2 | `landsat_tm_c2_l2` |
| Landsat 7 ETM+ Collection 2 Level 1 | `landsat_etm_c2_l1` |
| Landsat 7 ETM+ Collection 2 Level 2 | `landsat_etm_c2_l2` |
| Landsat 8/9 Collection 2 Level 1 | `landsat_ot_c2_l1` |
| Landsat 8/9 Collection 2 Level 2 | `landsat_ot_c2_l2` |
| Sentinel 2A | `sentinel_2a` |

(The Collection 1 datasets have been retired by the USGS.) Once logged in,
`rsgispy.dataaccess.usgs_m2m.get_earth_explorer_datasets(api_key)` lists the
datasets available to you.

You can also specify the min-max cloud cover thresholds, start and end time
periods and months of interest, alongside a point (`[lat, lon]`) or bounding
box (`[min_x, max_x, min_y, max_y]`) to be searched.

The function below can then be used to search for scenes:

In [3]:
# Specify the dataset which you want to download:
usgs_dataset = "landsat_ot_c2_l2"

In [ ]:
scn_lst, meta_data_dict = rsgispy.dataaccess.usgs_m2m.usgs_search(
    dataset=usgs_dataset,
    api_key=api_key,
    start_date=datetime.datetime(year=2020, month=4, day=1),
    end_date=datetime.datetime(year=2020, month=6, day=30),
    cloud_min=0,
    cloud_max=20,
    pt=[lat, lon],
    bbox=None,
    poly_geom=None,
    months=None,
    full_meta=False,
    max_n_rslts=10,
    start_n=None,
)

In [ ]:
scn_lst

## Get download IDs

The next step is to get a list of download IDs for the scenes, this is done
using the function below:

In [ ]:
scn_dsp_ids, scn_ent_ids = rsgispy.dataaccess.usgs_m2m.get_download_ids(scn_lst)

In [ ]:
scn_dsp_ids

## Create a scene list

Next you need to create a download list on the USGS server. You need to
provide a unique name for your list but also a time period for which the list
should be stored, as an ISO 8601 duration (see the `create_scene_list`
docstring). Generally I would specify that they should exist for 1 week,
which is `P1W`.

In [4]:
# Define a unique base name for the search and download list.
dnwld_name = "aber_r24_p204"

In [ ]:
scn_lst_add_info = rsgispy.dataaccess.usgs_m2m.create_scene_list(
    api_key=api_key,
    dataset=usgs_dataset,
    scn_ent_ids=scn_ent_ids,
    lst_name=f"{dnwld_name}_lst",
    lst_period="P1W",
)

## Check scenes are available for download

Once you have created the scene list, you need to get the download list:

In [ ]:
dwlds_lst = rsgispy.dataaccess.usgs_m2m.check_dwnld_opts(
    api_key=api_key,
    lst_name=f"{dnwld_name}_lst",
    dataset=usgs_dataset,
    dwnld_filetype="bundle",
    rm_lst=True,
)

## Request downloads

You can now request the download URLs:

In [ ]:
avail_dwn_urls, prep_dwnld_ids = rsgispy.dataaccess.usgs_m2m.request_downloads(
    api_key=api_key,
    dwlds_lst=dwlds_lst,
    dwnld_label=f"{dnwld_name}_dwnld",
)

## Get the list of URLs

The following loop extracts the URLs for each scene ID and checks whether the
scene is a tier 1 (T1) scene, as tier 2 (T2) data is commonly poorly
registered. This results in a dictionary of URLs to be downloaded:

In [ ]:
file_urls = dict()
for scn_id in scn_dsp_ids:
    print(scn_id)
    for url_id in avail_dwn_urls:
        if scn_id in avail_dwn_urls[url_id]:
            if "T1" in scn_id:
                file_urls[f"{scn_id}.tar"] = avail_dwn_urls[url_id]

In [ ]:
file_urls

## Create a database of the URLs to download

When there is a long list of URLs that need to be downloaded, it is useful to
keep track of whether a scene has been successfully downloaded. To do this,
rsgispy writes a small JSON database file listing each URL and whether it has
been downloaded.

*RSGISLib users:* RSGISLib used the `pysondb` module for this database;
rsgispy writes plain JSON, so `pysondb` is no longer needed.

In [5]:
ls_scns_file = os.path.join(out_dir, f"{dnwld_name}_scns.json")

In [ ]:
rsgispy.tools.http.create_file_listings_db(
    db_json=ls_scns_file,
    file_urls=file_urls,
)

## Log out of the USGS M2M system

Finally you should log out of the USGS M2M system using the function below:

In [ ]:
rsgispy.dataaccess.usgs_m2m.usgs_logout(api_key)

## Perform download

The final stage is to actually download the data requested. This is performed
using the following functions:

### Create output directory

In [6]:
out_dwnld_dir = os.path.join(out_dir, dnwld_name)
os.makedirs(out_dwnld_dir, exist_ok=True)

### Run download

rsgispy has a function which downloads the URLs within the JSON database and
records those which were successfully downloaded. You can read the database
file (`outputs/aber_r24_p204_scns.json`) in a text editor if you want to
check which files have been downloaded, or just run the function again and it
will only try to download the scenes which have not been previously
downloaded.

In [ ]:
rsgispy.tools.http.download_http_files_use_lst_db(
    db_json=ls_scns_file,
    out_dir_path=out_dwnld_dir,
    http_user=None,
    http_pass=None,
    use_wget=False,
    wget_time_out=60,
    check_file_exists=True,
)